# Becke Partition 一阶梯度简单理解

In [4]:
from pyscf import gto, dft, lib, grad, hessian, data
import numpy as np
from functools import partial

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [5]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [6]:
def get_grids(xyz):
    mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()
    grids = dft.grid.Grids(mol)
    grids.radii_adjust = dft.radi.becke_atomic_radii_adjust
    grids.build(sort_grids=False)
    return mol, grids

## PySCF 解析与数值导数验证

需要留意，PySCF 先前使用的 `grids_response_cc` 似乎是由于后来引入 padding 的问题，其结果我不太确定是否正确。目前使用的是 PySCF 中用于计算 VV10 导数所用到的函数。

In [7]:
xyz_0 = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""
xyz_p = """
N  0.0001   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""
xyz_m = """
N -0.0001   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

In [8]:
mol, grids = get_grids(xyz_0)

一阶解析格点导数计算如下：

In [9]:
dw = hessian.rks.get_dweight_dA(mol, grids)
dw.shape

(4, 3, 43328)

作为例子，一阶数值导数的第一个分量计算如下：

In [10]:
tmp_num_d = (get_grids(xyz_p)[1].weights - get_grids(xyz_m)[1].weights) / (2 * 0.0001 / data.nist.BOHR)

解析与数值导数有强一致。

In [11]:
np.allclose(dw[0, 0], tmp_num_d)

True

## Becke Partition 一阶梯度实现与公式对应

### 重新明确记号

为了方便公式推导与程序实现，我们需要重新定义一些记号。

- Becke 1988 原文的角标 $i, j$ 现在通常是指占据轨道。我们使用 $A, B, M, N$ 表示原子指标。
- 我们在可能的情况下，优先使用矩阵或张量记号。

### 一阶梯度：配分函数记号重新定义

$$
w_g = w_g^\text{quad} \frac{P_{M g}}{Z_g} \quad (g \in M)
$$

该公式是 Becke 原文 eq (22) 的重新表述。其中，$g \in M$ 是指，原子轨道的 DFT 格点 $g$ 是从 $M$ 原子的 Lebedev 网格中选取的。这在 PySCF (GPU4PySCF) 目前的实现中，是由 `grids.atm_idx` 给出的。需要留意，PySCF 现在的版本里引入了 grid padding，因此 `grids.atm_idx` 可能会包含一些 padding 的格点 (标记为 -1)。这些格点在 Becke 分区函数中不参与计算。

- $w_g$ 格点权重 `weights`，维度 $(g)$ `(ngrids,)`；
- $w_g^\text{quad}$ 原始格点权重 `quadrature_weights` (单个原子的 Lebedev 权重)，维度 $(g)$ `(ngrids,)`；
- $P_{M g}$ 原子 $M$ 的 Becke 分区函数，维度 $(M, g)$ `(natm, ngrids)`；
- $Z_g$ Becke 配分函数 (归一化因子)，维度 $(g)$ `(ngrids,)`；

    $$
    Z_g = \sum_N P_{N g}
    $$

我们在程序实现时，先不考虑 padding 的问题。对于 PySCF 的 padding，我个人的看法是不赞成 (是指我倾向于不在 REST 中实现类似的 padding)。padding 当然好，CPU 下能用 vmovapd，GPU 或许更重要。但这在我看来，既然格点权重被认为是不占用太多内存的量，复制一个不要紧，那么应该在算法层面上解决，而最好不要在接口层面。

一些重要的几何量记为

- 原子 $M$ 的坐标向量 $\bm{R}_M$，展开为三维分量 $R_{M t}$ `atm_coords`，维度 $(M, t)$ `(natm, 3)`；
- 格点坐标 $g$ 的坐标向量 $\bm{r}_g$，展开为三维分量 $r_{g t}$ `coords`，维度 $(g, t)$ `(ngrids, 3)`；

In [27]:
mask_nontab = grids.atm_idx >= 0      # just ignore the convention of PySCF padding
coords = grids.coords[mask_nontab]    # r [M, t] / (natm, 3)
weights = grids.weights[mask_nontab]  # w [g] / (ngrids,)
quadrature_weights = grids.quadrature_weights[mask_nontab]  # w^quad [g] / (ngrids,)
atm_idx = grids.atm_idx[mask_nontab]  # (ngrids,)
atm_coords = mol.atom_coords()        # R [M, t] / (natm, 3)

那么最初的导数公式可以写为

$$
\frac{\partial w_g}{\partial R_{A t}} = w_g^\text{quad} \frac{1}{Z_g} \frac{\partial P_{M g}}{\partial R_{A t}} - w_g^\text{quad} \frac{P_{M g}}{Z_g^2} \frac{\partial Z_g}{\partial R_{A t}}
$$

我们的目标是 $\partial_{R_{A t}} w_g$ `dw`，其维度为 $(A, g, t)$ `(natm, ngrids, 3)`。

### 一阶梯度：基本变量 $\mu$ 的导数

随后我们从底部开始推导。首先，我们记几何量

- $\Vert r \Vert_{A g}$ 原子 $A$ 到格点 $g$ 的距离 `grid_dist`，维度 $(A, g)$ `(natm, ngrids)`；

    $$
    \Vert r \Vert_{A g} = \sqrt{\sum_t (r_{g t} - R_{A t})^2}
    $$

- $\Vert R \Vert_{AB}$ 原子 $A$ 到原子 $B$ 的距离 `atom_dist`，维度 $(A, B)$ `(natm, natm)`；但需要留意，原子距离是在分母上的，因此对角线要设为 inf。

    $$
    \Vert R \Vert_{AB} = \sqrt{\sum_t (R_{B t} - R_{A t})^2}
    $$

在具体程序实现时，格点当然是分批实现的，不会一次性计算所有格点的距离矩阵。当然，我们为了公式表述与程序实现方便，这里就这么写了。

In [13]:
grid_dist = np.linalg.norm(coords[None, :, :] - atm_coords[:, None, :], axis=-1)       # |r| [M, g] / (natm, ngrids)
atom_dist = np.linalg.norm(atm_coords[:, None, :] - atm_coords[None, :, :], axis=-1)   # |R| [M, N] / (natm, natm)
for i in range(mol.natm):
    atom_dist[i, i] = np.inf  # avoid division by zero

Becke partition 的最基本变量是 $\mu_{ABg}$。但需要留意，这个量一般在实现时是作循环的，而非直接展开为张量。

$$
\mu_{ABg} = \frac{\Vert r \Vert_{A g} - \Vert r \Vert_{B g}}{\Vert R \Vert_{AB}}
$$

In [14]:
mu_tensor = (grid_dist[:, None, :] - grid_dist[None, :, :]) / atom_dist[:, :, None]  # μ [M, N, g] / (natm, natm, ngrids)
mu_tensor.shape

(4, 4, 43326)

现在我们考虑偏导数计算。需要注意，这些量只与原子 $A$ 和 $B$ 的坐标有关，而与其他原子无关。

下述两个量在程序中，只要分批得当，也可以预先存储下来，在程序表达上方便且高效。

- 格点距离导数 `d_grid_dist`，维度 $(A, t, g)$ `(natm, 3, ngrids)`：

    $$
    \Vert \partial r \Vert_{A t g} := \frac{\partial \Vert r \Vert_{A g}}{\partial R_{A t}} = \frac{R_{A t} - r_{g t}}{\Vert r \Vert_{A g}}
    $$

- 原子距离导数 `d_atom_dist`，维度 $(A, B, t)$ `(natm, natm, 3)`：

    $$
    \Vert \partial R \Vert_{A B t} := \frac{\partial \Vert R \Vert_{AB}}{\partial R_{A t}} = \frac{R_{A t} - R_{B t}}{\Vert R \Vert_{AB}}
    $$

    需要留意，从定义上看，$\Vert \partial R \Vert_{A B t} = - \Vert \partial R \Vert_{B A t}$。对 B 原子的导数是反对称的。

In [15]:
d_grid_dist = (atm_coords[:, :, None] - coords.T[None, :, :]) / grid_dist[:, None, :]    # d|r|/dR [M, t, g] / (natm, 3, ngrids)
d_atom_dist = (atm_coords[:, None, :] - atm_coords[None, :, :]) / atom_dist[:, :, None]  # d|R|/dR [M, N, t] / (natm, natm, 3)

那么 Becke partition 的基本变量 $\mu_{ABg}$ 的导数可以写为

$$
\begin{align}
\frac{\partial \mu_{ABg}}{\partial R_{A t}} &= \frac{1}{\Vert R \Vert_{AB}} \big( \Vert \partial r \Vert_{A t g} - \mu_{A B g} \Vert \partial R \Vert_{A B t} \big) \\
\frac{\partial \mu_{ABg}}{\partial R_{B t}} &= \frac{1}{\Vert R \Vert_{AB}} \big( - \Vert \partial r \Vert_{B t g} + \mu_{A B g} \Vert \partial R \Vert_{A B t} \big)
\end{align}
$$

留意该项的导数对 $A, B$ 以外的原子是零。

In [38]:
d_mu_tensor_A = (  d_grid_dist[:, None, :, :] - mu_tensor[:, :, None, :] * d_atom_dist[:, :, :, None]) / atom_dist[:, :, None, None]  # dμ/dR [M, N, t, g] / (natm, natm, 3, ngrids)
d_mu_tensor_B = (- d_grid_dist[None, :, :, :] + mu_tensor[:, :, None, :] * d_atom_dist[:, :, :, None]) / atom_dist[:, :, None, None]  # dμ/dR [M, N, t, g] / (natm, natm, 3, ngrids)

### 一阶梯度：截断特征函数 $s_3(\mu)$

到 $\mu_{ABg}$ 这一步，我们通常的做法是依单个格点 (也可能是 SIMD lane，但意图是差不多的)，那么我们就不用拘泥于其张量表示，而使用标量来说明问题。

我们回顾到

$$
\begin{align}
\tilde{s}(\mu) &= s_3 \circ \nu (\mu) \\
s_3(\nu) &= \frac{1}{2} (1 - f_3(\nu)) \tag{21} \\
f_3(\nu) &= p \circ p \circ p (\nu) \tag{20} \\
p(\nu) &= \frac{3}{2} \nu - \frac{1}{2} \nu^3 \tag{19} \\
\nu(\mu) &= \mu + a ( 1 - \mu^2 ) \tag{A2}
\end{align}
$$

上式除了 $a$ 不是关于 $\nu$ 的变量，其他都是。我们就按照非常普通的链式法则，给出 $s_3(\mu)$ 的导数。

$$
\begin{align}
\tilde{s}' &= s_3' \nu' \\
s_3' &= - \frac{1}{2} f_3' \\
f_3' &= p' (f_2) p' (f_1) p' \\
p' &= \frac{3}{2} (1 - \nu^2) \\
\nu' &= 1 - 2 a \mu
\end{align}
$$

In [17]:
fn_nu = lambda mu, a: mu + a * (1 - mu**2)
fn_p = lambda nu: 1.5 * nu - 0.5 * nu**3
fn_f1 = fn_p
fn_f2 = lambda nu: fn_p(fn_f1(nu))
fn_f3 = lambda nu: fn_p(fn_f2(nu))
fn_s = lambda nu: 0.5 * (1 - fn_f3(nu))
fn_s_full = lambda mu, a: fn_s(fn_nu(mu, a))

In [18]:
fn_d_nu = lambda mu, a: 1 - 2 * a * mu
fn_d_p = lambda nu: 1.5 * (1 - nu**2)
fn_d_f1 = fn_d_p
fn_d_f2 = lambda nu: fn_d_p(fn_f1(nu)) * fn_d_f1(nu)
fn_d_f3 = lambda nu: fn_d_p(fn_f2(nu)) * fn_d_f2(nu)
fn_d_s = lambda nu: -0.5 * fn_d_f3(nu)
fn_d_s_full = lambda mu, a: fn_d_s(fn_nu(mu, a)) * fn_d_nu(mu, a)

In [37]:
natm = mol.natm
ngrids = len(quadrature_weights)

becke_radii_adjust = grids.radii_adjust(mol, grids.atomic_radii)
fac_radii = [becke_radii_adjust(i, j, 0) for i in range(natm) for j in range(natm)]
fac_radii = np.array(fac_radii).reshape(natm, natm)

s_ = np.zeros([natm, natm, ngrids])
d_s = np.zeros([natm, natm, ngrids])

for i in range(natm):
    for j in range(natm):
        a = fac_radii[i, j]
        s_[i, j] = fn_s_full(mu_tensor[i, j], a)
        d_s[i, j] = fn_d_s_full(mu_tensor[i, j], a)

但这里需要留意，这里我们处理的是对 $\mu$ 的导数，而没有进一步涉及到 $\partial \mu / \partial R_{A t}$。这一步将在最后处理。

### 一阶梯度：分区函数 $P_{M g}$ 的导数

### 一阶梯度：开关函数值 $\tilde{s}$ 与对数导数核 $t$

由前一节我们已有 $\partial \tilde{s}_{MNg}/\partial \mu_{MNg}$ `d_s_full`。现在补上开关函数值本身，并构造对数导数核

$$
t_{MNg} = \frac{1}{\tilde{s}_{MNg}} \frac{\partial \tilde{s}_{MNg}}{\partial \mu_{MNg}}
$$

后续 $P_{Mg}$ 的连乘积导数将统一以 $t_{MNg}$ 表达。**数值正则化**：当 $\tilde{s}_{MNg}<10^{-14}$ (格点趋近 $N$ 原子) 时直接置 $t_{MNg}=0$，避免 $0\cdot\infty$，与 PySCF C 实现的 `inv()` 一致；该正则化对常规格点几乎不触发。

首先回顾定义：

$$
P_{M g} = \prod_{N \neq M} \tilde{s}_{M N g} \tag{13}
$$

连乘积导数是通过先取对数实现的：

$$
\begin{align}
\log P_{M g} &= \sum_{N \neq M} \log \tilde{s}_{M N g} \\
\partial \log P_{M g} &= \frac{1}{P_{M g}} \partial P_{M g} \\
\partial \sum_{N \neq M} \log \tilde{s}_{M N g} &= \sum_{N \neq M} \frac{1}{\tilde{s}_{M N g}} \partial \tilde{s}_{M N g} \\
\partial P_{M g} &= P_{M g} \sum_{N \neq M} \frac{1}{\tilde{s}_{M N g}} \partial \tilde{s}_{M N g}
\end{align}
$$

但需要留意，这里我们会遇到 $\tilde{s}_{M N g}$ 在分母的情况。如果 $\tilde{s}_{M N g}$ 的数值太小，则会影响计算的稳定性。在程序实现中，有必要对过小的 $\tilde{s}_{M N g}$ 直接置零。

In [ ]:
s_safe_mask = s_ > 1e-14
s = np.where(s_safe_mask, s_, 1.0)
d_log_s = np.where(s_safe_mask, d_s / s, 0.0)

for i in range(natm):
    s[i, i] = 1.0        # prod(N != M)
    d_log_s[i, i] = 0.0  # sum(N != M)

P = np.prod(s, axis=1)
Z = P.sum(axis=0)

In [ ]:
# 开关函数值 s̃_{MNg} = s_3(ν_{MNg})，以及对数导数核 t_{MNg} = (1/s̃) ds̃/dμ_{MNg}
a_fac = fac_radii[:, :, None]                                # a_{MN} (M, N, 1)
nu = fn_nu(mu_tensor, a_fac)                                 # ν_{MNg} (M, N, ngrids)
s_full = fn_s(nu)                                            # s̃_{MNg} (M, N, ngrids)

# 数值正则化：s̃<1e-14 时 t 置 0，避免 0*inf (与 PySCF C 的 inv() 一致)
s_safe = np.where(s_full > 1e-14, s_full, 1.0)
t_full = np.where(s_full > 1e-14, d_s / s_safe, 0.0)    # t_{MNg} (M, N, ngrids)
s_full.shape, t_full.shape

((4, 4, 43326), (4, 4, 43326))

In [22]:
# eq (13): P_{Mg} = ∏_{N≠M} s̃_{MNg}；自配对 N=M 不参与乘积，置 1
#          Z_g = Σ_M P_{Mg}
diag = np.eye(natm, dtype=bool)
s_for_prod = np.where(diag[:, :, None], 1.0, s_full)         # 对角置 1
t_for_sum = np.where(diag[:, :, None], 0.0, t_full)          # 对角置 0

P = np.prod(s_for_prod, axis=1)                              # P_{Mg} (M, ngrids)
Z = P.sum(axis=0)                                            # Z_g (ngrids,)
P.shape, Z.shape

((4, 43326), (43326,))

### 一阶梯度：$\partial Z_g/\partial R_{At}$ 与 $\partial P_{A_g,g}/\partial R_{At}$ 的张量化组装

把对数导数形式代入，$P_{Mg}$ 的导数为

$$
\frac{\partial P_{Mg}}{\partial R_{At}} = P_{Mg} \sum_{N\neq M} t_{MNg} \frac{\partial \mu_{MNg}}{\partial R_{At}}
$$

其中 $\partial \mu_{MNg}/\partial R_{At}$ 仅在 $A\in\{M,N\}$ 非零：$A=M$ 用 A-角色 `d_mu_tensor_A`，$A=N$ 用 B-角色 `d_mu_tensor_B`。据此：

- **$\partial Z_g/\partial R_{At} = \sum_M \partial P_{Mg}/\partial R_{At}$** 对所有原子 $A$ 直接计算 (两种角色都要)，这与 `dSigma/dR_G` 对应，不含关联原子特例。
- **$\partial P_{A_g,g}/\partial R_{At}$** ($A\neq A_g$) 仅 $A$ 取 B-角色的 $(A_g, A)$ 因子贡献；$A=A_g$ 行由平移不变性补齐 (与 C 实现跳过 `i_associated_atom == i_derivative_atom` 一致)。

In [23]:
# ∂Z_g/∂R_{At} = Σ_M ∂P_{Mg}/∂R_{At}，其中
#   ∂P_{Mg}/∂R_{At} = P_{Mg} Σ_{N≠M} t_{MNg} ∂μ_{MNg}/∂R_{At}
# ∂μ_{MNg}/∂R_{At} 仅在 A∈{M,N} 非零：A=M 用 A-角色 (d_mu_tensor_A)，A=N 用 B-角色 (d_mu_tensor_B)

# A-角色：A=M，∂P_A/∂R_A = P_A Σ_N t_{ANg} ∂μ_{ANg}/∂R_{At}   (A,N,g)·(A,N,t,g)->(A, t, g)
sumA = np.einsum("ang,antg->atg", t_for_sum, d_mu_tensor_A)
dP_dRA_self = P[:, None, :] * sumA                           # (A, t, g)

# B-角色：A=N，Σ_M P_{Mg} t_{MAg} ∂μ_{MAg}/∂R_{At}            (M,g)·(M,A,g)·(M,A,t,g)->(A, t, g)
sumB = np.einsum("mg,mag,matg->atg", P, t_for_sum, d_mu_tensor_B)

dZ_dR = dP_dRA_self + sumB                                   # ∂Z_g/∂R_{At} (A, t, g)
dZ_dR.shape

(4, 3, 43326)

In [24]:
# ∂P_{A_g,g}/∂R_{At} (A≠A_g)：A 取 B-角色，仅 (A_g, A) 因子贡献
#   = P_{A_g,g} t_{A_g A g} ∂μ_{A_g A g}/∂R_{At}
ar = np.arange(ngrids)
dmu_AgA = d_mu_tensor_B[atm_idx, :, :, ar]                   # (g, A, t) 选 M=A_g 行
t_AgA   = t_for_sum[atm_idx, :, ar]                          # (g, A)
P_Ag    = P[atm_idx, ar]                                     # (g,)

dPA_dR = P_Ag[:, None, None] * t_AgA[:, :, None] * dmu_AgA   # (g, A, t)
dPA_dR = dPA_dR.transpose(1, 2, 0)                           # (A, t, g)
# A==A_g 行：B-角色下 ∂μ=0、t=0，故自动为 0 (交由平移不变性补齐，与 C 实现一致)
dPA_dR.shape

(4, 3, 43326)

### 一阶梯度：组装 $\partial w_g/\partial R_{At}$ 与平移不变性

将 $\partial P_{A_g,g}$、$\partial Z_g$ 代入最初的导数公式

$$
\frac{\partial w_g}{\partial R_{At}} = w_g^\text{quad} \left[ \frac{1}{Z_g} \frac{\partial P_{A_g,g}}{\partial R_{At}} - \frac{P_{A_g,g}}{Z_g^2} \frac{\partial Z_g}{\partial R_{At}} \right]
$$

得到 $A\neq A_g$ 的各行。$A=A_g$ 行由平移不变性给出 (格点随关联原子运动，$\sum_A \partial w_g/\partial R_{At}=0$)：

$$
\frac{\partial w_g}{\partial R_{A_g t}} = -\sum_{A\neq A_g} \frac{\partial w_g}{\partial R_{At}}
$$

这与 `get_dweight_dA` 末尾 `dweight_dA[atm_idx, :, arange] = -sum(...)` 完全对应。

In [25]:
# ∂w_g/∂R_{At} = w_g^quad [ (1/Z_g) ∂P_{A_g,g}/∂R_{At} - (P_{A_g,g}/Z_g^2) ∂Z_g/∂R_{At} ]
dw = quadrature_weights[None, None, :] * (
    dPA_dR / Z[None, None, :]
    - (P_Ag / Z ** 2)[None, None, :] * dZ_dR
)                                                            # (A, t, g)，仅 A≠A_g 行有效

# eq (8): 平移不变性。C 实现跳过 A==A_g (置 0)，再由 -Σ_{A≠A_g} 补齐关联原子行
dw[atm_idx, :, ar] = 0.0
dw[atm_idx, :, ar] = -dw.sum(axis=0).T                       # (t, g).T -> (g, t)

dw.shape

(4, 3, 43326)

In [26]:
# 验证：与 PySCF 解析 get_dweight_dA 及数值导数比较
dw_ref = hessian.rks.get_dweight_dA(mol, grids)              # (natm, 3, ngrids_full) 含 padding

print("shape (mine, non-padding):", dw.shape)
print("vs get_dweight_dA  max abs diff:", np.max(np.abs(dw_ref[:, :, mask_nontab] - dw)))
print("vs get_dweight_dA  allclose   :", np.allclose(dw_ref[:, :, mask_nontab], dw))

# 数值导数 (扰动 N 原子 x 方向，格点随原子运动)
num_d = (get_grids(xyz_p)[1].weights - get_grids(xyz_m)[1].weights) / (2 * 0.0001 / data.nist.BOHR)
print("vs numerical [0,0] allclose   :", np.allclose(dw[0, 0], num_d[mask_nontab]))

shape (mine, non-padding): (4, 3, 43326)
vs get_dweight_dA  max abs diff: 3.552713678800501e-14
vs get_dweight_dA  allclose   : True
vs numerical [0,0] allclose   : True


# Old derivation

## $P_A$ 的对数导数

$P_A=\prod_{B\neq A}s_{AB}$ (10-1 eq 13)，其中 $s_{AB}=s_3(\nu_{AB})$ 仅通过 $\mu_{AB}$ 依赖原子坐标。由于格点固定，$\mu_{AB}=(r_A-r_B)/R_{AB}$ 只与端点原子 $\mathbf R_A,\mathbf R_B$ 有关。对乘积取对数导数，并引入对数导数核 $t_{AB}\equiv\frac{1}{s_{AB}}\frac{ds_{AB}}{d\mu_{AB}}$ (显式定义见 eq (5))：

$$
\frac{\partial P_A}{\partial \mathbf R_G}
= P_A\sum_{B\neq A}t_{AB}\,\frac{\partial \mu_{AB}}{\partial \mathbf R_G}.
\tag{2}
$$

求和中 $\partial\mu_{AB}/\partial\mathbf R_G$ 仅在 $G\in\{A,B\}$ 时非零 (见 eq (3))。这一对数导数形式正是 PySCF C 实现中 `switch_function_dmuds_over_s` 所计算的量，避免了把整个乘积 $P_A$ 直接对坐标差分的 $O(N_{\mathrm{atm}})$ 倍冗余。

## $\mu_{AB}$ 对原子坐标的导数

$\mu_{AB}=(r_A-r_B)/R_{AB}$，格点 $\mathbf r_g$ 固定。用到 $\partial r_A/\partial\mathbf R_A=\mathbf r_{Ag}/r_A$ (因 $r_A=|\mathbf r_g-\mathbf R_A|$)，以及 $\partial R_{AB}/\partial\mathbf R_A=\mathbf R_{AB}/R_{AB}$、$\partial R_{AB}/\partial\mathbf R_B=-\mathbf R_{AB}/R_{AB}$。对两个端点原子分别求导：

$$
\frac{\partial\mu_{AB}}{\partial\mathbf R_A}
= \frac{1}{R_{AB}}\left[\frac{\mathbf r_{Ag}}{r_A} - \mu_{AB}\frac{\mathbf R_{AB}}{R_{AB}}\right],
\tag{3a}
$$

$$
\frac{\partial\mu_{AB}}{\partial\mathbf R_B}
= \frac{1}{R_{AB}}\left[-\frac{\mathbf r_{Bg}}{r_B} + \mu_{AB}\frac{\mathbf R_{AB}}{R_{AB}}\right].
\tag{3b}
$$

对 $G\notin\{A,B\}$，$\partial\mu_{AB}/\partial\mathbf R_G=0$。式 (3a) 与 (3b) 中 $\mathbf R_{AB}$ 项符号相反，反映 $\mu_{AB}$ 对两端原子的反对称性；二者皆为 3 维向量 (xyz 分量)。后续把 $G$ 出现在乘积因子 $s_{XY}$ 中的两种角色记为：$G=X$ (A-角色，用 (3a)) 与 $G=Y$ (B-角色，用 (3b))。

## 开关函数 $s_{AB}$ 的导数与 $t_{AB}$

回顾 (10-1)：$\nu_{AB}=\mu_{AB}+a_{AB}(1-\mu_{AB}^2)$ (eq A2)，$p(\mu)=\frac32\mu-\frac12\mu^3$ (eq 19)，$f_3=p\circ p\circ p$ (eq 20)，$s_{AB}=\frac12(1-f_3(\nu_{AB}))$ (eq 21)。$p'(x)=\frac32(1-x^2)$。记 $f_1=p(\nu_{AB})$，$f_2=p(f_1)$，$f_3=p(f_2)$，$d\nu_{AB}/d\mu_{AB}=1-2a_{AB}\mu_{AB}$。由链式法则：

$$
\frac{ds_{AB}}{d\mu_{AB}} = -\frac12\,p'(f_2)\,p'(f_1)\,p'(\nu_{AB})\,\frac{d\nu_{AB}}{d\mu_{AB}}.
\tag{4}
$$

代入 eq (2) 中引用的对数导数核：

$$
t_{AB} = \frac{1}{s_{AB}}\frac{ds_{AB}}{d\mu_{AB}}.
\tag{5}
$$

**数值正则化**：当格点趋近 $B$ 原子时 $s_{AB}\to 0$、$t_{AB}\to\infty$，但乘积 $P_B\,t_{BG}$ 保持有限 ($P_B$ 含因子 $s_{BG}$)。为避免 $0\cdot\infty$，实现中对 $s_{AB}<10^{-14}$ 直接置 $t_{AB}=0$，与 PySCF C 实现的 `inv()` 一致；对常规格点该正则化几乎不触发。

## 组装 $\partial P_{A_g}/\partial\mathbf R_G$ 与 $\partial\Sigma/\partial\mathbf R_G$

对 $G\neq A_g$ (关联原子情形由 eq (8) 处理)，eq (2) 中仅 $B=G$ 的因子有贡献 ($G$ 取 B-角色)，用 (3b)：

$$
\frac{\partial P_{A_g}}{\partial \mathbf R_G}
= P_{A_g}\,t_{A_g G}\,\frac{\partial\mu_{A_g G}}{\partial\mathbf R_G}, \qquad G\neq A_g.
\tag{6}
$$

$\Sigma=\sum_B P_B$ 的导数按 $G$ 在各 $P_B$ 乘积中的两种角色拆分：$G$ 作为 A-角色出现在 $P_G$ 的每个因子 $s_{GC}$ ($C\neq G$，用 (3a))；$G$ 作为 B-角色出现在每个 $P_B$ ($B\neq G$) 的因子 $s_{BG}$ 中 (用 (3b))：

$$
\frac{\partial\Sigma}{\partial \mathbf R_G}
= \underbrace{P_G\sum_{C\neq G}t_{GC}\frac{\partial\mu_{GC}}{\partial\mathbf R_G}}_{\partial P_G/\partial\mathbf R_G\;(\text{eq 3a})}
\;+\; \underbrace{\sum_{B\neq G}P_B\,t_{BG}\,\frac{\partial\mu_{BG}}{\partial\mathbf R_G}}_{\text{eq 3b}}.
\tag{7}
$$

将 (6)、(7) 代入 (1) 即得 $G\neq A_g$ 的 $\partial w_g/\partial\mathbf R_G$。$\partial\Sigma/\partial\mathbf R_G$ 不依赖 $A_g$，可对所有格点统一向量化计算；$\partial P_{A_g}/\partial\mathbf R_G$ 则按每格点的 $A_g$ 取对应行。

## 平移不变性与效率

格点跟随原子运动，故权重只依赖相对位置，$\sum_G\partial w_g/\partial\mathbf R_G=0$。据此，关联原子 $A_g$ 的行无需显式求导，由剩余和给出：

$$
\frac{\partial w_g}{\partial\mathbf R_{A_g}} = -\sum_{G\neq A_g}\frac{\partial w_g}{\partial\mathbf R_G}.
\tag{8}
$$

这与 `get_dweight_dA` 末尾的 `dweight_dA[atm_idx, :, arange] = -sum(...)` 完全对应。padding 格点 (`atm_idx<0`) 权重为 0，导数亦为 0。

**效率**：$s_{AB},P_B,\Sigma$ 每格点一次 $O(N_{\mathrm{atm}}^2)$ 预计算；对每个导数原子 $G$，eq (7) 的两个求和与 eq (6) 均为 $O(N_{\mathrm{atm}})$ 并向量化于格点，总复杂度 $O(N_g\,N_{\mathrm{atm}}^2)$。相比 C 实现在每个 $G$ 内重算全部 $P_B$ 的 $O(N_g\,N_{\mathrm{atm}}^3)$，此处预计算 $P_B$ 消除了冗余 (代价是 $O(N_{\mathrm{atm}}^2)$ 的 $s_{AB}$ 存储，对常见体系可忽略)。

In [23]:
def becke_weight_derivative(grid_coords, grid_weights, atm_coords, a_factor, atm_idx):
    """核心求值函数 (纯 ndarray，对应 PySCF C 的 ``VXCbecke_weight_derivative``，但无 ctypes)。

    输入 (与 C 入参一一对应)：
        grid_coords  (N, 3)   格点 r_g (求导时固定)
        grid_weights (N,)     原始权重 w_g^bare  (C: grid_quadrature_weights)
        atm_coords   (M, 3)   原子坐标 R_A
        a_factor     (M, M)   radii 矫正表 a_{AB}
        atm_idx      (N,)     关联原子 A_g (int；<0 表示 padding)

    输出：
        dwdA (M, 3, N)  即 dw_g/dR_G，仅填充 G != A_g (及非 padding) 的行；
        G == A_g 的行留 0，交由接口层以平移不变性补齐 (eq (8))，与
        ``get_dweight_dA`` 中 C 调用后做 ``dweight_dA[atm_idx,...] = -sum(...)``
        的分工完全一致。

    公式 tag 见本 notebook 各 markdown 单元。
    """
    N = grid_coords.shape[0]
    M = atm_coords.shape[0]
    arangeN = np.arange(N)

    # --- 几何量 ---
    # r_{Ag} = R_A - r_g,  r_A = |r_{Ag}|
    rAg_vec = atm_coords[:, None, :] - grid_coords[None, :, :]      # (M, N, 3)
    rA = np.linalg.norm(rAg_vec, axis=2)                            # (M, N)
    rA_safe = np.where(rA > 1e-14, rA, 1.0)                         # inv() 风格安全除
    # R_{AB} = R_A - R_B
    Rab_vec = atm_coords[:, None, :] - atm_coords[None, :, :]       # (M, M, 3)
    Rab = np.linalg.norm(Rab_vec, axis=2)                            # (M, M)
    diag = np.eye(M, dtype=bool)
    Rab[diag] = 1.0                                                  # 避免 0/0
    Rab_inv = 1.0 / Rab
    Rab_inv[diag] = 0.0
    a_factor = np.array(a_factor, copy=True)
    a_factor[diag] = 0.0

    # eq (11): mu_{AB} = (r_A - r_B)/R_{AB} ;  eq (A2): nu_{AB} = mu + a (1 - mu^2)
    mu_AB = (rA[:, None, :] - rA[None, :, :]) * Rab_inv[:, :, None]   # (M, M, N)
    nu_AB = mu_AB + a_factor[:, :, None] * (1.0 - mu_AB ** 2)          # (M, M, N)

    # eq (19,20): f3 = p(p(p(nu))) ;  eq (21): s_{AB} = 1/2 (1 - f3)
    def p(x):
        return 1.5 * x - 0.5 * x ** 3                                  # eq (19)
    f1 = p(nu_AB)
    f2 = p(f1)
    f3 = p(f2)                                                          # eq (20)
    s_AB = 0.5 * (1.0 - f3)                                             # eq (21), (M, M, N)

    # eq (4): ds_{AB}/dmu_{AB} = -1/2 p'(f2) p'(f1) p'(nu) dnu/dmu,  p'(x)=3/2(1-x^2)
    p1 = 1.5 * (1.0 - nu_AB ** 2)
    p2 = 1.5 * (1.0 - f1 ** 2)
    p3 = 1.5 * (1.0 - f2 ** 2)
    dnu_dmu = 1.0 - 2.0 * a_factor[:, :, None] * mu_AB
    ds_dmu = -0.5 * p3 * p2 * p1 * dnu_dmu                             # (M, M, N)
    # eq (5): t_{AB} = (1/s) ds/dmu，配合 inv() 正则化 (s<1e-14 置 0)
    s_safe = np.where(s_AB > 1e-14, s_AB, 1.0)
    t_AB = np.where(s_AB > 1e-14, ds_dmu / s_safe, 0.0)

    # 排除自配对 (B=A)：eq (13) 乘积只对 B!=A
    s_AB[diag] = 1.0
    t_AB[diag] = 0.0

    # eq (13): P_A = prod_{B!=A} s_{AB} ;  Sigma = sum_B P_B
    P = np.prod(s_AB, axis=1)                                           # (M, N)
    Sigma = P.sum(axis=0)                                               # (N,)

    dwdA = np.zeros((M, 3, N))                                          # dwdA[G, xyz, g]

    for G in range(M):
        uG = rAg_vec[G] / rA_safe[G][:, None]                          # (N,3) r_{Gg}/r_G

        # ---- eq (7): dSigma/dR_G ----
        # (a) G 取 A-角色，因子 s_{GC} (C!=G)，eq (3a)
        mu_GC = mu_AB[G]                                                # (M, N) over C
        t_GC = t_AB[G]                                                  # (M, N)
        RabG = Rab[G]                                                   # (M,)
        RabG_vec = Rab_vec[G]                                           # (M, 3)
        dmu_GC = (Rab_inv[G][:, None, None]                             # (M, N, 3)
                  * (uG[None, :, :]
                     - mu_GC[:, :, None] * RabG_vec[:, None, :] / RabG[:, None, None]))
        dP_G = P[G][:, None] * np.einsum("cg,cgi->gi", t_GC, dmu_GC)   # dP_G/dR_G, (N,3)

        # (b) G 取 B-角色，因子 s_{BG} (B!=G)，eq (3b)
        mu_BG = mu_AB[:, G, :]                                          # (M, N) over B
        t_BG = t_AB[:, G, :]                                            # (M, N)
        RabBG = Rab[:, G]                                               # (M,)
        RabBG_vec = Rab_vec[:, G, :]                                    # (M, 3)
        dmu_BG = (Rab_inv[:, G][:, None, None]                          # (M, N, 3)
                  * (-uG[None, :, :]
                     + mu_BG[:, :, None] * RabBG_vec[:, None, :] / RabBG[:, None, None]))
        sumB = np.einsum("bg,bgi->gi", P * t_BG, dmu_BG)               # (N, 3)

        dSigma_dR_G = dP_G + sumB                                       # eq (7), (N, 3)

        # ---- eq (6): dP_{A_g}/dR_G  (G != A_g)，pair (A_g, G)，G 取 B-角色 ----
        dmu_AgG = dmu_BG[atm_idx, arangeN, :]                          # (N, 3) 选 B=A_g 行
        t_AgG = t_BG[atm_idx, arangeN]                                  # (N,)
        P_Ag = P[atm_idx, arangeN]                                      # (N,)
        dPA_dR_G = P_Ag[:, None] * t_AgG[:, None] * dmu_AgG            # eq (6), (N, 3)

        mask = (atm_idx != G) & (atm_idx >= 0)                          # G != A_g 且非 padding
        dPA_dR_G[~mask] = 0.0

        # ---- eq (1): dw_g/dR_G ----
        dw_G = grid_weights[:, None] * (
            dPA_dR_G / Sigma[:, None]
            - P_Ag[:, None] / Sigma[:, None] ** 2 * dSigma_dR_G
        )                                                               # (N, 3)
        dw_G[~mask] = 0.0
        dwdA[G] = dw_G.T                                                # (3, N)

    return dwdA


def becke_partition_dweight_dA(mol, grids):
    """接口函数 (对应 ``pyscf.hessian.rks.get_dweight_dA``)：从 mol/grids 抽取
    纯数组，调用核心函数，再以平移不变性补齐关联原子行。

    返回 dwdA[G, xyz, g]，形状 (natm, 3, ngrids)。
    """
    natm = mol.natm
    grid_coords = np.asarray(grids.coords, order="C")              # (N, 3)
    grid_weights = np.asarray(grids.quadrature_weights)            # (N,)
    atm_idx = np.asarray(grids.atm_idx)                            # (N,)
    atm_coords = np.asarray(mol.atom_coords(), order="C")         # (M, 3)

    # --- radii 矫正表 a_{AB} (与 get_dweight_dA 构造方式完全一致) ---
    radii_adjust = grids.radii_adjust
    atomic_radii = grids.atomic_radii
    if callable(radii_adjust) and atomic_radii is not None:
        f_adj = radii_adjust(mol, atomic_radii)
        a_factor = np.array([f_adj(i, j, 0)
                             for i in range(natm) for j in range(natm)]
                            ).reshape(natm, natm)
    else:
        a_factor = np.zeros((natm, natm))

    # 核心求值 (G != A_g 的行)
    dwdA = becke_weight_derivative(grid_coords, grid_weights, atm_coords,
                                   a_factor, atm_idx)

    # ---- eq (8): 平移不变性填充关联原子行 ----
    arangeN = np.arange(grids.coords.shape[0])
    dwdA[atm_idx, 0, arangeN] = -np.sum(dwdA[:, 0, :], axis=0)
    dwdA[atm_idx, 1, arangeN] = -np.sum(dwdA[:, 1, :], axis=0)
    dwdA[atm_idx, 2, arangeN] = -np.sum(dwdA[:, 2, :], axis=0)
    return dwdA

In [24]:
# 验证：与 PySCF get_dweight_dA (解析) 及数值导数比较
dw_ref = hessian.rks.get_dweight_dA(mol, grids)
dw_mine = becke_partition_dweight_dA(mol, grids)

print("shape:", dw_ref.shape, dw_mine.shape)
print("vs get_dweight_dA  max abs diff:", np.max(np.abs(dw_ref - dw_mine)))
print("vs get_dweight_dA  allclose   :", np.allclose(dw_ref, dw_mine))

# 数值导数 (沿用本 notebook 开头的 xyz_0/xyz_p/xyz_m，仅扰动 N 原子 x 方向)
num_d = (get_grids(xyz_p)[1].weights - get_grids(xyz_m)[1].weights) / (2 * 0.0001 / data.nist.BOHR)
print("vs numerical [0,0] allclose   :", np.allclose(dw_mine[0, 0], num_d))

shape: (4, 3, 43328) (4, 3, 43328)
vs get_dweight_dA  max abs diff: 2.1316282072803006e-14
vs get_dweight_dA  allclose   : True


vs numerical [0,0] allclose   : True
